# 02. Feature Engineering

Final feature set used by the deliverable: 49 anchor columns plus 12 leakage-safe historical priors, 61 columns in total.

## Anchor columns (49)

- 19 raw numeric columns covering visitor history, property quality, price, search context, and the `random_bool` flag.
- 24 raw competitor columns: `comp{1..8}_rate`, `comp{1..8}_inv`, `comp{1..8}_rate_percent_diff`.
- 5 categorical id columns: `site_id`, `visitor_location_country_id`, `prop_country_id`, `prop_id`, `srch_destination_id`.
- 1 derived flag: `has_visitor_history` = 1 when both `visitor_hist_starrating` and `visitor_hist_adr_usd` are non-null.
- Train-only columns dropped: `position`, `click_bool`, `booking_bool`, `gross_bookings_usd`.

LightGBM is told to treat the five id columns as categorical via `categorical_feature`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.features import expected_feature_columns, NUMERIC_COLS, COMP_COLS, CATEGORICAL_COLS, DERIVED_COLS
len(expected_feature_columns()), len(NUMERIC_COLS), len(COMP_COLS), len(CATEGORICAL_COLS), len(DERIVED_COLS)

## Hypothesis-driven features that did not survive

We tested five hypothesis blocks on the locked holdout, with same-seed deltas against the anchor. None met the +0.0011 acceptance threshold; all were rejected.

| hypothesis block | description | mean same-seed delta | decision |
|---|---|---:|---|
| Per-search normalisation | per-search z-score and rank on price, star, location | -0.000327 | reject |
| Leave-one-out property priors | per-`prop_id` LOO click-rate and conversion-rate | -0.000525 | reject |
| Mean-position proxy | per-`prop_id` mean position from `random_bool=0` rows | +0.000545 | reject (sub-noise) |
| Negative downsampling | downsampling 1:1 / 1:4 of negatives | -0.016920 / -0.005600 | reject |
| Competitor and price composites | competitor aggregates and Liu et al. composites | near zero | reject |

Interpretation: in our anchor `prop_id` is already a categorical feature, so the LightGBM trees learn per-property biases directly from the id; explicit per-property aggregates duplicate that signal. Earlier prior work (Bellucci 2021) gained from these features partly because their anchor did not include `prop_id` as categorical.

## Final added features: leakage-safe historical priors

Three blocks of four features each, twelve features in total:

- `prop_*`: aggregated by `prop_id`.
- `dest_*`: aggregated by `srch_destination_id`.
- `prop_dest_*`: aggregated by `(prop_id, srch_destination_id)`.

Each block contains: `*_impressions_log1p`, `*_click_rate_smooth`, `*_booking_rate_smooth`, `*_relevance_mean_smooth`.

Smoothing is Laplace style:
$$
\widehat{r}_g \;=\; \frac{\sum_{i\in g} y_i + \alpha\,\bar{y}_{\text{global}}}{|g| + \alpha}, \qquad \alpha=20.
$$

Unseen groups fall back to the global rate via the same identity (when $|g|=0$ the smoothed rate equals $\bar{y}_{\text{global}}$).

## Why K-fold instead of leave-one-out

For training rows we use 5-fold group K-fold on `srch_id` with `fold_seed=42`. A row in fold $f$ gets its priors from the other 4 folds, never from the same query. This guards against two leakage modes at once: same-row leakage and same-search leakage. For validation, holdout, and test rows we use the full training slice, which is leakage-safe because those queries are group-disjoint from train by construction.

Naive train+test joint encoding (a tactic in some 2013 Kaggle solutions) is forbidden, since it leaks test labels into train statistics.

In [ ]:
from src.prior_features import PRIOR_BLOCKS, PRIOR_STAT_NAMES, prior_feature_columns
PRIOR_BLOCKS, PRIOR_STAT_NAMES, prior_feature_columns()

## Negative-result feature block (D)

We also tested a fourth block keyed by `(prop_id, visitor_location_country_id)` with four additional features, bringing the total to 65. The same-seed holdout delta versus the prior 3-block model was +0.000745, below the +0.0010 acceptance threshold. The block was rejected and is not in the final pipeline. Top D-block feature importance was `prop_visitor_country_click_rate_smooth` with gain about 12,319, roughly six times weaker than the strongest C-block feature; the signal mostly overlapped with the existing prop and prop_dest priors.